

# Public transport assignment with Optimal Strategies

In this example, we import a GTFS feed to our model, create a public transport network, create project match connectors, and perform a Spiess & Florian assignment. [Click here](https://doi.org/10.1016/0191-2615(89)90034-9) 
to check out the article.

We use data from Coquimbo, a city in La Serena Metropolitan Area in Chile.


.. admonition:: References

  * `transit_assignment`



.. seealso::
    Several functions, methods, classes and modules are used in this example:

    * :func:`aequilibrae.transit.Transit`
    * :func:`aequilibrae.transit.TransitGraphBuilder`
    * :func:`aequilibrae.paths.TransitClass`
    * :func:`aequilibrae.paths.TransitAssignment`
    * :func:`aequilibrae.matrix.AequilibraeMatrix`



In [1]:
# Imports for example construction
from uuid import uuid4
from os.path import join
from tempfile import gettempdir

from aequilibrae.transit import Transit
from aequilibrae.utils.create_example import create_example

from aequilibrae.paths.public_transport import HyperpathGenerating

In [2]:
# Let's create an empty project on an arbitrary folder.

##### CHANGE TEMP DIR #####
examples_dir = 'temp_examples'
#examples_dir = gettempdir()
fldr = join(examples_dir, uuid4().hex)
project = create_example(fldr, "coquimbo")

Let's create our ``Transit`` object.



In [3]:
data = Transit(project)

## Graph building
Let's build the transit network. We'll disable ``outer_stop_transfers`` and ``walking_edges`` 
because Coquimbo doesn't have any parent stations.

For the OD connections we'll use the ``overlapping_regions`` method and create some accurate line geometry later.
Creating the graph should only take a moment. By default zoning information is pulled from the project network. 
If you have your own zoning information add it using ``graph.add_zones(zones)`` then ``graph.create_graph()``. 



In [4]:
graph = data.create_graph(with_outer_stop_transfers=False, with_walking_edges=False, blocking_centroid_flows=False, connector_method="overlapping_regions")


# We drop geometry here for the sake of display.
graph.vertices.drop(columns="geometry")

,node_id,node_type,stop_id,line_id,line_seg_idx,taz_id
index,,,,,,
0,1,od,,,-1,1
1,2,od,,,-1,2
2,3,od,,,-1,3
3,4,od,,,-1,4
4,5,od,,,-1,5
...,...,...,...,...,...,...
362,363,alighting,10000000075,1_10001003000,31,
363,364,alighting,10000000076,1_10001003000,32,
364,365,alighting,10000000077,1_10001003000,33,


In [5]:
graph.edges

,link_id,link_type,line_id,stop_id,line_seg_idx,b_node,a_node,trav_time,freq,o_line_id,d_line_id,direction
index,,,,,,,,,,,,
0,1,on-board,1_10001001000,,0,212,290,86400.000000,inf,,,1
1,2,on-board,1_10001001000,,1,213,291,86400.000000,inf,,,1
2,3,on-board,1_10001001000,,2,214,292,86400.000000,inf,,,1
3,4,on-board,1_10001001000,,3,215,293,86400.000000,inf,,,1
4,5,on-board,1_10001001000,,4,216,294,86400.000000,inf,,,1
...,...,...,...,...,...,...,...,...,...,...,...,...
641,642,egress_connector,,,-1,166,112,630.543431,inf,,,1
642,643,egress_connector,,,-1,177,112,754.081137,inf,,,1
643,644,egress_connector,,,-1,178,112,386.262027,inf,,,1


In [6]:
# Spiess & Florian
sf = HyperpathGenerating(
    graph.edges, tail="b_node", head="a_node", trav_time="trav_time", freq="freq"
)

In [7]:
sf.run(origin=58, destination=75, volume=1.0)

In [8]:
temp = sf._edges
temp[temp.volume != 0].edge_idx.values

array([ 22,  24,  26,  28, 100, 102, 104, 106, 178, 180, 182, 184, 311,
       353, 366, 374, 378, 410, 422, 434, 440, 455, 466, 478, 532, 533,
       539, 549, 586, 601, 611, 613, 620, 638], dtype=int64)